In [ ]:
import pandas as pd
import requests

cities_and_districts = [
    "Delhi",
    "Mumbai",
    "Kolkata",
    "Bengaluru",
    "Chennai",
    "Hyderabad",
    "Ahmedabad", "Ahmadabad", 
    "Pune",
    "Lucknow",
    "Ernakulam",          
    "Jaipur",
    "Patna",
    "Visakhapatnam",
    "Bhopal",
    "Kamrup Metropolitan" 
]

pattern = '|'.join(cities_and_districts)

resource_ids = [
    "4d0fae21-fdec-4922-88ce-771a3df78de8",
    "3945dd91-4b4b-489b-8032-f1d78f34129b",
    "b453f280-75e2-47d9-9258-2b3c8b947513",
    "72276380-d9a7-4cbc-b359-c67439529582",
    "35729797-b783-4ace-9c7e-69aa8eec52ae",
    "f9420380-03ad-45c8-a7b3-d09db8e238c3",
    "c89a5d03-dfcb-4524-96f6-6843c9d4f140"
]

print("Initializing 28-day Meta Movement Distribution Extraction...")
dataframes = []

for rid in resource_ids:
    meta_url = f"https://data.humdata.org/api/3/action/resource_show?id={rid}"

    try:
        response = requests.get(meta_url)
        response_data = response.json()

        if response_data.get("success"):
            csv_url = response_data["result"]["url"]
            print(f"Fetching raw CSV for resource: {rid} ...")

            df_raw = pd.read_csv(csv_url)

            if 'country' in df_raw.columns:
                df_ind = df_raw[df_raw['country'] == 'IND'].copy()
            else:
                df_ind = df_raw.copy()

            df_filtered = df_ind[df_ind['gadm_name'].str.contains(pattern, case=False, na=False)].copy()

            if not df_filtered.empty:
                dataframes.append(df_filtered)
                print(f"  -> Success! Extracted {len(df_filtered)} relevant rows.")
            else:
                print(f"  -> Warning: No 15-city matches found in this specific file.")

        else:
            print(f"  -> API Error: Could not locate metadata for {rid}")

    except Exception as e:
        print(f"  -> ERROR processing resource {rid}: {e}")

if len(dataframes) > 0:
    print("\nAggregating 28-day trailing average...")
    master_df = pd.concat(dataframes, ignore_index=True)

    # Group by the GADM name and Distance Category to mathematically average the ping fraction
    avg_df = master_df.groupby(
        ['country', 'gadm_name', 'home_to_ping_distance_category']
    )['distance_category_ping_fraction'].mean().reset_index()

    output_filename = "15_cities_mobility_raw.csv"
    avg_df.to_csv(output_filename, index=False)

    print(f"\n✅ Pipeline complete! Total averaged rows generated: {len(avg_df)}")
    print(f"File saved as: {output_filename}")

    print("\nMatched District Names in Final Dataset:")
    print(avg_df['gadm_name'].unique())
else:
    print("\nCRITICAL FAILURE: No data was extracted. Check API connectivity.")

Initializing 28-day Meta Movement Distribution Extraction...
Fetching raw CSV for resource: 4d0fae21-fdec-4922-88ce-771a3df78de8 ...
  -> Success! Extracted 224 relevant rows.
Fetching raw CSV for resource: 3945dd91-4b4b-489b-8032-f1d78f34129b ...
  -> Success! Extracted 224 relevant rows.
Fetching raw CSV for resource: b453f280-75e2-47d9-9258-2b3c8b947513 ...
  -> Success! Extracted 224 relevant rows.
Fetching raw CSV for resource: 72276380-d9a7-4cbc-b359-c67439529582 ...
  -> Success! Extracted 224 relevant rows.
Fetching raw CSV for resource: 35729797-b783-4ace-9c7e-69aa8eec52ae ...
  -> Success! Extracted 232 relevant rows.
Fetching raw CSV for resource: f9420380-03ad-45c8-a7b3-d09db8e238c3 ...
  -> Success! Extracted 248 relevant rows.
Fetching raw CSV for resource: c89a5d03-dfcb-4524-96f6-6843c9d4f140 ...
  -> Success! Extracted 244 relevant rows.

Aggregating 28-day trailing average...

✅ Pipeline complete! Total averaged rows generated: 56
File saved as: 15_cities_mobility_raw.

In [ ]:

import argparse
import math
import pandas as pd
import numpy as np

GADM_TO_CITY = {
    "Kamrup Metropolitan":   "Guwahati",
    "Ernakulam":             "Kochi",
    "Chennai":               "Chennai",
    "Bhopal":                "Bhopal",
    "Mumbai Suburban":       "Mumbai",
    "Mumbai City":           "Mumbai",
    "Pune":                  "Pune",
    "Lucknow":               "Lucknow",
    "Visakhapatnam":         "Visakhapatnam",
    "Jaipur":                "Jaipur",
    "Kolkata":               "Kolkata",
    "Patna":                 "Patna",
    "Ahmadabad":             "Ahmedabad",
    "Ahmedabad":             "Ahmedabad",
    "Hyderabad":             "Hyderabad",
    "Delhi":                 "Delhi",
    "Bengaluru":             "Bengaluru",
}

CITIES = {
    "Delhi":         {"pop": 33_127_402, "lat": 28.7041, "lon": 77.1025},
    "Mumbai":        {"pop": 23_582_050, "lat": 19.0760, "lon": 72.8777},
    "Kolkata":       {"pop": 24_384_528, "lat": 22.5726, "lon": 88.3639},
    "Bengaluru":     {"pop": 13_678_383, "lat": 12.9716, "lon": 77.5946},
    "Chennai":       {"pop": 11_362_949, "lat": 13.0827, "lon": 80.2707},
    "Hyderabad":     {"pop":  9_790_908, "lat": 17.3850, "lon": 78.4867},
    "Ahmedabad":     {"pop":  7_649_898, "lat": 23.0225, "lon": 72.5714},
    "Pune":          {"pop":  7_926_450, "lat": 18.5204, "lon": 73.8567},
    "Lucknow":       {"pop":  5_228_335, "lat": 26.8467, "lon": 80.9462},
    "Kochi":         {"pop":  3_870_022, "lat":  9.9312, "lon": 76.2673},
    "Jaipur":        {"pop":  4_039_465, "lat": 26.9124, "lon": 75.7873},
    "Patna":         {"pop":  5_175_312, "lat": 25.5941, "lon": 85.1376},
    "Visakhapatnam": {"pop":  1_695_716, "lat": 17.6868, "lon": 83.2185},
    "Bhopal":        {"pop":  2_451_628, "lat": 23.2599, "lon": 77.4126},
    "Guwahati":      {"pop":  1_349_253, "lat": 26.1445, "lon": 91.7362},
}

NH_CORRIDORS = {
    ("Delhi",          "Jaipur"):           ("NE4",         "trunk"),
    ("Delhi",          "Ahmedabad"):        ("NE4",         "trunk"),
    ("Delhi",          "Mumbai"):           ("NE4",         "trunk"),
    ("Jaipur",         "Ahmedabad"):        ("NE4",         "trunk"),
    ("Ahmedabad",      "Mumbai"):           ("NE4",         "trunk"),
    ("Mumbai",         "Pune"):             ("NH48",        "trunk"),
    ("Pune",           "Bengaluru"):        ("NH48",        "primary"),
    ("Mumbai",         "Bengaluru"):        ("NH48",        "primary"),
    ("Mumbai",         "Chennai"):          ("NH48",        "primary"),
    ("Delhi",          "Bhopal"):           ("NH44",        "trunk"),
    ("Delhi",          "Hyderabad"):        ("NH44",        "trunk"),
    ("Delhi",          "Bengaluru"):        ("NH44",        "trunk"),
    ("Delhi",          "Chennai"):          ("NH44",        "trunk"),
    ("Bhopal",         "Hyderabad"):        ("NH44",        "primary"),
    ("Hyderabad",      "Bengaluru"):        ("NH44",        "trunk"),
    ("Bengaluru",      "Chennai"):          ("NE7",         "trunk"),
    ("Delhi",          "Lucknow"):          ("NE3",         "trunk"),
    ("Delhi",          "Patna"):            ("NH19",        "primary"),
    ("Delhi",          "Kolkata"):          ("NH19",        "primary"),
    ("Lucknow",        "Patna"):            ("NH19/NH27",   "primary"),
    ("Lucknow",        "Kolkata"):          ("NH19",        "primary"),
    ("Patna",          "Kolkata"):          ("NH19",        "primary"),
    ("Lucknow",        "Guwahati"):         ("NH27",        "primary"),
    ("Patna",          "Guwahati"):         ("NH27",        "primary"),
    ("Kolkata",        "Guwahati"):         ("NH27",        "primary"),
    ("Ahmedabad",      "Bhopal"):           ("NH27/NH47",   "primary"),
    ("Kolkata",        "Visakhapatnam"):    ("NH16",        "primary"),
    ("Visakhapatnam",  "Chennai"):          ("NH16",        "primary"),
    ("Visakhapatnam",  "Hyderabad"):        ("NH16/NH65",   "primary"),
    ("Kochi",          "Bengaluru"):        ("NH544",       "primary"),
    ("Kochi",          "Chennai"):          ("NH544/NH44",  "primary"),
    ("Pune",           "Hyderabad"):        ("NH65",        "primary"),
    ("Mumbai",         "Hyderabad"):        ("NH65",        "primary"),
    ("Mumbai",         "Kochi"):            ("NH66",        "primary"),
}

MULTIPLIER = {
    "trunk":   1.35,
    "primary": 1.15,
    "none":    0.85,
}


ANCHOR_SOURCE        = "Mumbai"
ANCHOR_TARGET        = "Pune"
ANCHOR_DAILY_PAX     = 300_000


def haversine_km(lat1, lon1, lat2, lon2):
    R    = 6371.0
    p1   = math.radians(lat1)
    p2   = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a    = (math.sin(dphi / 2) ** 2
            + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))


def load_meta_fractions(csv_path: str) -> dict:
    df = pd.read_csv(csv_path)
    df = df[df["country"] == "IND"].copy()
    df["city"] = df["gadm_name"].map(GADM_TO_CITY)
    df = df[df["city"].notna()].copy()
    df = df[
        df["home_to_ping_distance_category"].isin(["[10, 100)", "100+"])
    ].copy()
    df["distance_category_ping_fraction"] = df[
        "distance_category_ping_fraction"
    ].clip(lower=0)

    grouped = (
        df.groupby(["city", "home_to_ping_distance_category"])[
            "distance_category_ping_fraction"
        ]
        .mean()
        .reset_index()
    )

    fractions = {}
    for city, grp in grouped.groupby("city"):
        row = grp.set_index("home_to_ping_distance_category")[
            "distance_category_ping_fraction"
        ].to_dict()
        fractions[city] = {
            "near": row.get("[10, 100)", 0.0),
            "far":  row.get("100+",      0.0),
        }
    return fractions


def build_integrated_matrix(fractions: dict):
    names = list(CITIES.keys())
    n     = len(names)
    pops  = np.array([CITIES[c]["pop"] for c in names], dtype=float)

    nh_lookup = {}
    for (a, b), val in NH_CORRIDORS.items():
        nh_lookup[(a, b)] = val
        nh_lookup[(b, a)] = val

    mean_near = (
        np.mean([v["near"] for v in fractions.values()]) if fractions else 0.04
    )
    mean_far = (
        np.mean([v["far"]  for v in fractions.values()]) if fractions else 0.008
    )

    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                dist[i, j] = haversine_km(
                    CITIES[names[i]]["lat"], CITIES[names[i]]["lon"],
                    CITIES[names[j]]["lat"], CITIES[names[j]]["lon"],
                )

    
    raw_flows = np.zeros((n, n))

    for i in range(n):
        m_i = pops[i]

        for j in range(n):
            if i == j:
                continue

            d_ij = dist[i, j]

            
            if fractions and names[i] in fractions:
                mob_rate = (
                    fractions[names[i]]["near"] if d_ij < 100
                    else fractions[names[i]]["far"]
                )
            else:
                mob_rate = mean_near if d_ij < 100 else mean_far

           
            corridor_width_km = max(50.0, 0.08 * d_ij)
            corridor_area_km2 = d_ij * corridor_width_km
            INDIA_POP_DENSITY = 430.0   # persons / km²
            synthetic_S_ij    = corridor_area_km2 * INDIA_POP_DENSITY

            S_ij = synthetic_S_ij + sum(
                pops[k]
                for k in range(n)
                if k != i and k != j and dist[i, k] < d_ij
            )

            n_j    = pops[j]
            denom1 = m_i + S_ij
            denom2 = m_i + n_j + S_ij

            if denom1 > 0 and denom2 > 0:
                beta             = 0.002
                distance_penalty = math.exp(-beta * d_ij)

                
                rad_weight       = (m_i * n_j) / (denom1 * denom2)
                raw_flows[i, j]  = mob_rate * m_i * rad_weight * distance_penalty

   
    mumbai_idx    = names.index(ANCHOR_SOURCE)
    pune_idx      = names.index(ANCHOR_TARGET)
    anchor_raw    = raw_flows[mumbai_idx, pune_idx]

    if anchor_raw <= 0:
        raise ValueError(
            f"Raw flow for {ANCHOR_SOURCE}→{ANCHOR_TARGET} is zero. "
            "Check distance friction or population data."
        )

    anchor_nh_mult = MULTIPLIER.get(
    nh_lookup.get((ANCHOR_SOURCE, ANCHOR_TARGET), ("none", "none"))[1], 0.85
    )
    C = ANCHOR_DAILY_PAX / (anchor_raw * anchor_nh_mult)
    total_raw  = raw_flows.sum()

    print(f"  Anchor edge ({ANCHOR_SOURCE}→{ANCHOR_TARGET}) raw : {anchor_raw:.6f}")
    print(f"  Calibration constant C               : {C:.6f}")
    print(f"  Network total daily travelers        : {total_raw * C:,.0f}")
    print(f"  (MoRTH national total not used — "
          f"single-edge anchor applied instead)")

    rows_out = []
    for i in range(n):
        row_sum_nh = sum(
            raw_flows[i, j] * MULTIPLIER.get(
                nh_lookup.get((names[i], names[j]), ("none", "none"))[1], 0.85
            )
            for j in range(n) if i != j
        )

        for j in range(n):
            if i == j:
                continue

            calibrated_flow = raw_flows[i, j] * C

            pair        = (names[i], names[j])
            nh_no, cls  = nh_lookup.get(pair, ("none", "none"))
            nh_mult     = MULTIPLIER.get(cls, 0.85)
            norm        = (
                raw_flows[i, j] * nh_mult / row_sum_nh
                if row_sum_nh > 0 else 0.0
            )

            rows_out.append({
                "source_node_id":                names[i],
                "target_node_id":                names[j],
                "haversine_distance_km":         round(dist[i, j], 3),
                "raw_daily_travelers": round(calibrated_flow * nh_mult, 2),
                "normalized_terrestrial_weight": round(norm, 6),
                "highway_class":                 cls,
                "road_class_multiplier":         nh_mult,
                "matrix_method":                 "integrated_meta_nh_radiation",
            })

    return pd.DataFrame(rows_out)


def run_sanity_checks(df: pd.DataFrame):
    print("\n── Sanity Checks ─────────────────────────────────────────────────")

    mum_pune = df[
        (df.source_node_id == ANCHOR_SOURCE) &
        (df.target_node_id == ANCHOR_TARGET)
    ]["raw_daily_travelers"].values[0]
    print(f"  Anchor corridor (Mumbai→Pune) : {mum_pune:>12,.0f}")
    print(f"  Target                        : {ANCHOR_DAILY_PAX:>12,}")
    anchor_ok = abs(mum_pune - ANCHOR_DAILY_PAX) < 1_000
    print(f"  {'✅  Anchor held' if anchor_ok else '❌  Anchor drifted — check C calculation'}")

    del_mum = df[
        (df.source_node_id == "Delhi") & (df.target_node_id == "Mumbai")
    ]["raw_daily_travelers"].values[0]
    del_guw = df[
        (df.source_node_id == "Delhi") & (df.target_node_id == "Guwahati")
    ]["raw_daily_travelers"].values[0]
    print(f"\n  Delhi→Mumbai  (1153 km) : {del_mum:>12,.0f}  (expect < 300,000)")
    print(f"  Delhi→Guwahati (1900 km): {del_guw:>12,.0f}  (expect < Delhi→Mumbai)")
    decay_ok = del_guw < del_mum < mum_pune
    print(f"  {'✅  Distance decay monotonic' if decay_ok else '⚠️  Decay not monotonic — check beta'}")

    
    threshold = ANCHOR_DAILY_PAX * 3
    dominated = df[df["raw_daily_travelers"] > threshold][
        ["source_node_id", "target_node_id", "raw_daily_travelers"]
    ]
    if dominated.empty:
        print(f"\n  ✅  No edge exceeds 3× anchor ({threshold:,}) — network is balanced")
    else:
        print(f"\n  ⚠️  Edges exceeding 3× anchor ({threshold:,}):")
        print(dominated.to_string(index=False))

    benchmarks = [
        ("Delhi",     "Mumbai"),
        ("Mumbai",    "Pune"),
        ("Delhi",     "Jaipur"),
        ("Delhi",     "Guwahati"),
        ("Bengaluru", "Chennai"),
    ]
    print("\n  Benchmark corridors:")
    for src, tgt in benchmarks:
        val = df[
            (df.source_node_id == src) & (df.target_node_id == tgt)
        ]["raw_daily_travelers"].values
        v    = val[0] if len(val) else float("nan")
        flag = "  ⚠️  EXCEEDS 3× ANCHOR" if v > threshold else ""
        print(f"    {src:>15} → {tgt:<15}  {v:>12,.0f}{flag}")

    print("──────────────────────────────────────────────────────────────────\n")


def main():
    import sys

    parser = argparse.ArgumentParser(
        description="Build single-edge-anchored radiation mobility matrix (15 cities)."
    )
    parser.add_argument(
        "--input", required=True,
        help="Path to 15_cities_mobility_raw.csv (Meta Movement Distribution)"
    )

    if "ipykernel" in sys.modules:
        args = parser.parse_args(args=["--input", "15_cities_mobility_raw.csv"])
    else:
        args = parser.parse_args()

    print("Loading 28-day Meta fractions …")
    fractions = load_meta_fractions(args.input)
    print(f"  Cities mapped with Meta data : {len(fractions)}")
    print(f"  Anchor edge                  : {ANCHOR_SOURCE} → {ANCHOR_TARGET}"
          f" = {ANCHOR_DAILY_PAX:,} daily passengers")

    print("\nBuilding single-edge-anchored Radiation Matrix …")
    final_df = build_integrated_matrix(fractions)

    run_sanity_checks(final_df)

    out_file = "meta_mobility_edges.csv"
    final_df.to_csv(out_file, index=False)
    print(f"✅  Wrote {out_file}  ({len(final_df)} edges)")
    print(
        f"   Delhi→Mumbai          : "
        f"{final_df[(final_df.source_node_id=='Delhi')    & (final_df.target_node_id=='Mumbai')   ]['raw_daily_travelers'].values[0]:>14,.0f}"
    )
    print(
        f"   Mumbai→Pune           : "
        f"{final_df[(final_df.source_node_id=='Mumbai')   & (final_df.target_node_id=='Pune')     ]['raw_daily_travelers'].values[0]:>14,.0f}"
    )
    print(
        f"   Bengaluru→Chennai     : "
        f"{final_df[(final_df.source_node_id=='Bengaluru')& (final_df.target_node_id=='Chennai')  ]['raw_daily_travelers'].values[0]:>14,.0f}"
    )
    print(
        f"   Delhi→Guwahati        : "
        f"{final_df[(final_df.source_node_id=='Delhi')    & (final_df.target_node_id=='Guwahati') ]['raw_daily_travelers'].values[0]:>14,.0f}"
    )
    print(
        f"   Total across all edges: "
        f"{final_df['raw_daily_travelers'].sum():>14,.0f}"
    )


if __name__ == "__main__":
    main()

Loading 28-day Meta fractions …
  Cities mapped with Meta data : 13
  Anchor edge                  : Mumbai → Pune = 300,000 daily passengers

Building single-edge-anchored Radiation Matrix …
  Anchor edge (Mumbai→Pune) raw : 14297.498286
  Calibration constant C               : 15.542735
  Network total daily travelers        : 2,497,961
  (MoRTH national total not used — single-edge anchor applied instead)

── Sanity Checks ─────────────────────────────────────────────────
  Anchor corridor (Mumbai→Pune) :      300,000
  Target                        :      300,000
  ✅  Anchor held

  Delhi→Mumbai  (1153 km) :       33,469  (expect < 300,000)
  Delhi→Guwahati (1900 km):          209  (expect < Delhi→Mumbai)
  ✅  Distance decay monotonic

  ✅  No edge exceeds 3× anchor (900,000) — network is balanced

  Benchmark corridors:
              Delhi → Mumbai                 33,469
             Mumbai → Pune                  300,000
              Delhi → Jaipur                290,848
       

In [ ]:
df = pd.read_csv("meta_mobility_edges.csv")

print(df[(df.source_node_id=='Mumbai') & (df.target_node_id=='Pune')]['raw_daily_travelers'].values[0])

print(df[(df.source_node_id=='Delhi') & (df.target_node_id=='Mumbai')]['raw_daily_travelers'].values[0])

threshold = 273_800_000 * 0.70 * 0.05
print(df[df['raw_daily_travelers'] > threshold][['source_node_id','target_node_id','raw_daily_travelers']])

300000.0
33468.62
Empty DataFrame
Columns: [source_node_id, target_node_id, raw_daily_travelers]
Index: []


In [ ]:
import pandas as pd

old_file = 'meta_mobility_edgesold.csv'  
new_file = 'meta_mobility_edges.csv'

old_df = pd.read_csv(old_file)
new_df = pd.read_csv(new_file)

compare_df = pd.merge(
    old_df[['source_node_id', 'target_node_id', 'haversine_distance_km', 'raw_daily_travelers']],
    new_df[['source_node_id', 'target_node_id', 'raw_daily_travelers']],
    on=['source_node_id', 'target_node_id'],
    suffixes=('_old', '_new')
)

compare_df['volume_change'] = compare_df['raw_daily_travelers_new'] - compare_df['raw_daily_travelers_old']
compare_df['percent_change'] = (compare_df['volume_change'] / compare_df['raw_daily_travelers_old']) * 100

pd.options.display.float_format = '{:,.0f}'.format

compare_df = compare_df.sort_values(by='haversine_distance_km', ascending=False)

print("=== LONGEST ROUTES (Expect massive percentage drops) ===")
print(compare_df[['source_node_id', 'target_node_id', 'haversine_distance_km',
                  'raw_daily_travelers_old', 'raw_daily_travelers_new', 'percent_change']].head(10))

print("\n=== SHORTEST ROUTES (Expect volume to remain structurally heavy) ===")
print(compare_df[['source_node_id', 'target_node_id', 'haversine_distance_km',
                  'raw_daily_travelers_old', 'raw_daily_travelers_new', 'percent_change']].tail(10))

print("\n=== BENCHMARK CORRIDOR CHECK ===")
benchmarks = compare_df[
    ((compare_df['source_node_id'] == 'Delhi') & (compare_df['target_node_id'] == 'Mumbai')) |
    ((compare_df['source_node_id'] == 'Mumbai') & (compare_df['target_node_id'] == 'Pune'))
]
print(benchmarks[['source_node_id', 'target_node_id', 'raw_daily_travelers_old', 'raw_daily_travelers_new']])

=== LONGEST ROUTES (Expect massive percentage drops) ===
    source_node_id target_node_id  haversine_distance_km  \
205       Guwahati          Kochi                  2,429   
139          Kochi       Guwahati                  2,429   
8            Delhi          Kochi                  2,089   
126          Kochi          Delhi                  2,089   
197       Guwahati         Mumbai                  2,086   
27          Mumbai       Guwahati                  2,086   
55       Bengaluru       Guwahati                  2,080   
199       Guwahati      Bengaluru                  2,080   
203       Guwahati           Pune                  2,022   
111           Pune       Guwahati                  2,022   

     raw_daily_travelers_old  raw_daily_travelers_new  percent_change  
205                        8                        0             -99  
139                       41                        0            -100  
8                      7,230                      120             

In [ ]:
import pandas as pd
import numpy as np

new = pd.read_csv("meta_mobility_edges.csv")

print("Before:")
print(f"Total daily travelers: {new['raw_daily_travelers'].sum():,.0f}")

MAJOR_CORRIDORS = [
    ("Delhi", "Mumbai"),
    ("Delhi", "Jaipur"),
    ("Mumbai", "Pune"),
    ("Bengaluru", "Chennai"),
    ("Mumbai", "Ahmedabad"),
    ("Delhi", "Lucknow"),
    ("Bengaluru", "Hyderabad"),
    ("Kolkata", "Patna"),
    # Probably include
    ("Delhi", "Ahmedabad"),
    ("Mumbai", "Hyderabad"),
    ("Chennai", "Hyderabad"),
    ("Lucknow", "Patna"),
    ("Delhi", "Patna"),
]

whitelist = {frozenset(pair) for pair in MAJOR_CORRIDORS}

df = new.copy()

patched = []
for pair in whitelist:
    cities = list(pair)
    a, b = cities[0], cities[1]

    mask_ab = (df["source_node_id"] == a) & (df["target_node_id"] == b)
    mask_ba = (df["source_node_id"] == b) & (df["target_node_id"] == a)

    if mask_ab.sum() == 0 or mask_ba.sum() == 0:
        # try reverse
        mask_ab = (df["source_node_id"] == b) & (df["target_node_id"] == a)
        mask_ba = (df["source_node_id"] == a) & (df["target_node_id"] == b)

    if mask_ab.sum() == 0 or mask_ba.sum() == 0:
        print(f"  WARNING: could not find both directions for {a} <-> {b}")
        continue

    val_ab = df.loc[mask_ab, "raw_daily_travelers"].values[0]
    val_ba = df.loc[mask_ba, "raw_daily_travelers"].values[0]
    max_val = max(val_ab, val_ba)

    df.loc[mask_ab, "raw_daily_travelers"] = max_val
    df.loc[mask_ba, "raw_daily_travelers"] = max_val

    patched.append({
        "corridor": f"{a} <-> {b}",
        "old_fwd": round(val_ab, 0),
        "old_rev": round(val_ba, 0),
        "max_applied": round(max_val, 0),
        "which_won": f"{a}->{b}" if val_ab >= val_ba else f"{b}->{a}"
    })

patch_df = pd.DataFrame(patched).sort_values("max_applied", ascending=False)
print("\n=== CORRIDORS PATCHED ===")
print(patch_df.to_string(index=False))

print(f"\nAfter:")
print(f"Total daily travelers: {df['raw_daily_travelers'].sum():,.0f}")
print(f"Delta: +{df['raw_daily_travelers'].sum() - new['raw_daily_travelers'].sum():,.0f}")

df.to_csv("meta_mobility_edges_bidirectional.csv", index=False)
print("\nSaved to meta_mobility_edges_bidirectional.csv")

Before:
Total daily travelers: 3,372,247

=== CORRIDORS PATCHED ===
               corridor  old_fwd  old_rev  max_applied            which_won
        Mumbai <-> Pune  300,000  402,497      402,497         Pune->Mumbai
  Chennai <-> Bengaluru  288,146  325,716      325,716   Bengaluru->Chennai
       Delhi <-> Jaipur  290,848  172,140      290,848        Delhi->Jaipur
      Delhi <-> Lucknow  172,341  155,430      172,341       Delhi->Lucknow
   Ahmedabad <-> Mumbai  124,373   68,396      124,373    Ahmedabad->Mumbai
      Kolkata <-> Patna   82,709   57,884       82,709       Kolkata->Patna
Bengaluru <-> Hyderabad   58,595   77,536       77,536 Hyderabad->Bengaluru
    Ahmedabad <-> Delhi    9,364   62,866       62,866     Delhi->Ahmedabad
  Chennai <-> Hyderabad   43,660   17,903       43,660   Chennai->Hyderabad
      Lucknow <-> Patna    6,481   41,487       41,487       Patna->Lucknow
   Mumbai <-> Hyderabad   37,061   15,659       37,061    Mumbai->Hyderabad
       Delhi <-> Mum

In [ ]:
import pandas as pd

old_file = 'meta_mobility_edges.csv'  # Replace with the name of your old output file
new_file = 'meta_mobility_edges_bidirectional.csv'

old_df = pd.read_csv(old_file)
new_df = pd.read_csv(new_file)

compare_df = pd.merge(
    old_df[['source_node_id', 'target_node_id', 'haversine_distance_km', 'raw_daily_travelers']],
    new_df[['source_node_id', 'target_node_id', 'raw_daily_travelers']],
    on=['source_node_id', 'target_node_id'],
    suffixes=('_old', '_new')
)

compare_df['volume_change'] = compare_df['raw_daily_travelers_new'] - compare_df['raw_daily_travelers_old']
compare_df['percent_change'] = (compare_df['volume_change'] / compare_df['raw_daily_travelers_old']) * 100

pd.options.display.float_format = '{:,.0f}'.format

compare_df = compare_df.sort_values(by='haversine_distance_km', ascending=False)

print("=== LONGEST ROUTES (Expect massive percentage drops) ===")
print(compare_df[['source_node_id', 'target_node_id', 'haversine_distance_km',
                  'raw_daily_travelers_old', 'raw_daily_travelers_new', 'percent_change']].head(10))

print("\n=== SHORTEST ROUTES (Expect volume to remain structurally heavy) ===")
print(compare_df[['source_node_id', 'target_node_id', 'haversine_distance_km',
                  'raw_daily_travelers_old', 'raw_daily_travelers_new', 'percent_change']].tail(10))

print("\n=== BENCHMARK CORRIDOR CHECK ===")
benchmarks = compare_df[
    ((compare_df['source_node_id'] == 'Delhi') & (compare_df['target_node_id'] == 'Mumbai')) |
    ((compare_df['source_node_id'] == 'Mumbai') & (compare_df['target_node_id'] == 'Pune'))
]
print(benchmarks[['source_node_id', 'target_node_id', 'raw_daily_travelers_old', 'raw_daily_travelers_new']])

=== LONGEST ROUTES (Expect massive percentage drops) ===
    source_node_id target_node_id  haversine_distance_km  \
205       Guwahati          Kochi                  2,429   
139          Kochi       Guwahati                  2,429   
8            Delhi          Kochi                  2,089   
126          Kochi          Delhi                  2,089   
197       Guwahati         Mumbai                  2,086   
27          Mumbai       Guwahati                  2,086   
55       Bengaluru       Guwahati                  2,080   
199       Guwahati      Bengaluru                  2,080   
203       Guwahati           Pune                  2,022   
111           Pune       Guwahati                  2,022   

     raw_daily_travelers_old  raw_daily_travelers_new  percent_change  
205                        0                        0               0  
139                        0                        0               0  
8                        120                      120             